# Datathon 2026 – Task 1: Traffic Speed Forecasting
**Approach**: Per-road Ridge Regression with lag features + text incident features
- Input: 15 historical speed observations per road
- Output: Speed predictions at h5, h10, h15 (5/10/15 min ahead) for 1260 roads
- Algorithm: Ridge regression — fast, no installation needed, works on Kaggle CPU kernel


In [ ]:
import numpy as np
import pandas as pd
import json, os, time
from pathlib import Path

# ─── Paths ────────────────────────────────────────────────────────────────────
# Adjust INPUT_DIR to match your Kaggle dataset name
INPUT_DIR = Path('/kaggle/input')
# Try to find the dataset directory automatically
candidates = [d for d in INPUT_DIR.iterdir() if d.is_dir()] if INPUT_DIR.exists() else []
if candidates:
    DATA = candidates[0]
    print(f'Using dataset: {DATA}')
else:
    DATA = Path('.')  # fallback for local run

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
t0 = time.time()
print('Setup done')


In [ ]:
# ─── Load Data ────────────────────────────────────────────────────────────────
print('Loading training speed data...')
speed_m1 = np.load(DATA / 'train' / 'train_speed_m1_1_11160.npy')  # (11160, 1260)
speed_m2 = np.load(DATA / 'train' / 'train_speed_m2_1_5039.npy')   # (5039,  1260)
train = np.vstack([speed_m1, speed_m2]).astype(np.float32)          # (16199, 1260)

print('Loading test data...')
test_X = np.load(DATA / 'test' / 'test_X_hist.npy').astype(np.float32)  # (540, 15, 1260)

print('Loading text (incident) data...')
with open(DATA / 'train' / 'train_text_m1_1_11160.json') as f:
    text_m1 = json.load(f)   # dict: 'm1_1' -> str
with open(DATA / 'train' / 'train_text_m2_1_5039.json') as f:
    text_m2 = json.load(f)
with open(DATA / 'test' / 'test_texts.json') as f:
    test_texts = json.load(f)  # dict: 'test_00000' -> str

T, R = train.shape
H, F = 15, 3
n_win = T - H - F + 1
N_test = test_X.shape[0]
print(f'train: {train.shape}, test_X: {test_X.shape}, n_windows: {n_win}')
print(f'Loaded in {time.time()-t0:.1f}s')


In [ ]:
# ─── Text Feature Extraction ───────────────────────────────────────────────────
# Extract simple binary features from incident text
KEYWORDS = ['road closure', 'construction', 'accident', 'prohibit', 'announcement']

def text_feats(text: str) -> np.ndarray:
    """Returns binary feature vector for incident text."""
    text = text.lower()
    feats = [float(kw in text) for kw in KEYWORDS]
    feats.append(text.count('.'))  # number of incidents
    return np.array(feats, dtype=np.float32)

N_TEXT = len(KEYWORDS) + 1

# Map training text: ordered by timestep
# text_m1 keys: 'm1_1' to 'm1_11160' → index 0..11159
# text_m2 keys: 'm2_1' to 'm2_5039' → index 11160..16198
train_text_feats = np.zeros((T, N_TEXT), dtype=np.float32)
for i in range(speed_m1.shape[0]):
    key = f'm1_{i+1}'
    if key in text_m1:
        train_text_feats[i] = text_feats(text_m1[key])
for i in range(speed_m2.shape[0]):
    key = f'm2_{i+1}'
    if key in text_m2:
        train_text_feats[speed_m1.shape[0]+i] = text_feats(text_m2[key])

# Test text features
test_text_feats = np.zeros((N_test, N_TEXT), dtype=np.float32)
for i in range(N_test):
    key = f'test_{i:05d}'
    if key in test_texts:
        test_text_feats[i] = text_feats(test_texts[key])

print(f'Text features: {N_TEXT} per timestep')
print(f'Train text feats: {train_text_feats.shape}')
print(f'Test text feats: {test_text_feats.shape}')


In [ ]:
# ─── Build Model via Cross-Correlations (no huge array materialisation) ────────
# For each road r independently, we solve:
#   W_r = argmin_W || X_r W - y_r ||^2 + lambda ||W||^2
# where X_r has rows [speed[t, r], ..., speed[t+H-1, r], text_feat[t], ..., 1]
# and y_r has rows [speed[t+H, r], speed[t+H+1, r], speed[t+H+2, r]]
#
# Key: XtX and Xty can be computed via element-wise cross-products over time
# without materialising the full (n_win × n_features × R) array.

lam = 1.0   # Ridge regularisation

# Feature columns:
# 0..H-1:   speed lags (offset 0 to H-1) — per road
# H..H+N_TEXT-1: text features at target timestep (shared across roads)
# H+N_TEXT:  bias
N_FEAT = H + N_TEXT + 1

print(f'Building XtX/Xty for all roads... (N_FEAT={N_FEAT})')

# Text and bias are shared across roads — precompute once as (n_win, N_TEXT+1)
# The 'target' timestep for window starting at t is t+H (we use text at t+H)
shared = np.concatenate([
    train_text_feats[H:H+n_win],        # (n_win, N_TEXT)
    np.ones((n_win, 1), dtype=np.float32)  # bias
], axis=1)  # (n_win, N_TEXT+1)

# XtX: (R, N_FEAT, N_FEAT),  Xty: (R, N_FEAT, F)
XtX = np.zeros((R, N_FEAT, N_FEAT), dtype=np.float64)
Xty = np.zeros((R, N_FEAT, F), dtype=np.float64)

# ── Speed lag blocks (rows/cols 0..H-1) — road-specific ──────────────────────
for i in range(H):
    a = train[i:i+n_win]  # (n_win, R)  — speed lag i
    for j in range(i, H):
        b = train[j:j+n_win]
        v = (a * b).sum(axis=0)   # (R,) dot product over time
        XtX[:, i, j] = v;  XtX[:, j, i] = v
    # Speed lag i vs text/bias features
    # a: (n_win, R), shared: (n_win, N_TEXT+1)
    cross_ts = a.T @ shared   # (R, N_TEXT+1)  — dot products
    XtX[:, i, H:] = cross_ts;  XtX[:, H:, i] = cross_ts
    # Speed lag i vs targets
    for f in range(F):
        Xty[:, i, f] = (a * train[H+f:H+f+n_win]).sum(axis=0)

# ── Shared (text/bias) block (rows/cols H..N_FEAT-1) ─────────────────────────
# shared: (n_win, N_TEXT+1) — same for all roads
shared_block = shared.T @ shared  # (N_TEXT+1, N_TEXT+1)
XtX[:, H:, H:] = shared_block[None]  # broadcast over roads

# Shared vs targets: (N_TEXT+1, n_win) @ (n_win, R) = (N_TEXT+1, R)
for f in range(F):
    Xty[:, H:, f] = (shared.T @ train[H+f:H+f+n_win]).T   # (R, N_TEXT+1)

# ── Add ridge regularisation ──────────────────────────────────────────────────
XtX += lam * np.eye(N_FEAT, dtype=np.float64)[None]

print(f'XtX/Xty built in {time.time()-t0:.1f}s')


In [ ]:
# ─── Solve for weights and predict ────────────────────────────────────────────
W = np.linalg.solve(XtX, Xty).astype(np.float32)  # (R, N_FEAT, F)
print(f'W: {W.shape} | {time.time()-t0:.1f}s')

# Build test feature matrix: (N_test, N_FEAT, R)
# Speed lags: test_X (540, 15, 1260) → dim 0=sample, 1=lag, 2=road
# Text: test_text_feats (540, N_TEXT)
# Bias: ones
test_feat = np.concatenate([
    test_X,  # (540, 15, 1260) — speed lags
    np.repeat(test_text_feats[:, :, None], R, axis=2).astype(np.float32),  # (540, N_TEXT, 1260)
    np.ones((N_test, 1, R), dtype=np.float32)   # (540, 1, 1260)
], axis=1)  # (540, N_FEAT, 1260)

# Predict: pred[n, f, r] = sum_h test_feat[n, h, r] * W[r, h, f]
pred = np.einsum('nhr,rhf->nfr', test_feat, W)   # (540, F, 1260)
pred = np.clip(pred, 0.0, 200.0)
print(f'pred: {pred.shape} | mean={pred.mean():.2f} | {time.time()-t0:.1f}s')


In [ ]:
# ─── Generate Submission CSV ───────────────────────────────────────────────────
horizons = [5, 10, 15]
ids, spd = [], []
for si in range(N_test):
    s = f'test_{si:05d}'
    for fi, h in enumerate(horizons):
        for ri in range(R):
            ids.append(f'{s}_h{h}_r{ri}')
            spd.append(float(pred[si, fi, ri]))

submission = pd.DataFrame({'id': ids, 'speed': spd})
out_path = OUT / 'submission.csv'
submission.to_csv(out_path, index=False)

print(f'Submission saved to {out_path}')
print(f'Total rows: {len(submission)}')
print(submission.head(10))
print(f'Speed stats:\n{submission["speed"].describe()}')
print(f'Total time: {time.time()-t0:.1f}s')


## Summary

**Model**: Per-road Ridge Regression (closed-form, no gradient descent)

**Features per sample**:
- 15 historical speed values for the target road (lag-1 to lag-15)
- 5 binary text/incident features (road closure, construction, accident, prohibit, announcement)
- 1 incident count feature
- 1 bias term
→ **22 features total**

**Training**: 16,199 timesteps × 1,260 roads → ~16,182 sliding windows per road

**Inference**: predicts h5, h10, h15 simultaneously for all 1,260 roads × 540 test samples